In [1]:
# B-1 master table  ->  data/meta/master.csv
import os, json, shutil
from pathlib import Path
import numpy as np, pandas as pd

root = Path.cwd()
while not (root / "data" / "imgs_cleaned").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
CLEAN = Path("data/imgs_cleaned"); A1 = Path("logs/preprocess_a1"); META = Path("data/meta")

# logs/ is git-ignored, so keep the two files the build depends on under data/meta
for n in ("letterbox.csv", "black_boxes.csv"): shutil.copy(A1 / n, META / n)

a1 = pd.read_csv(A1 / "measurements.csv")[["file", "cls", "source", "w", "h", "bytes", "sha1",
                                           "bright", "contrast", "blur", "chroma", "flags"]]
near = pd.read_csv(META / "near_dup_groups.csv")[["file", "label", "group_id", "nn_sim", "nn_file"]]
rv = pd.read_csv(META / "review.csv")
drop = set(rv.loc[rv.action == "drop", "file"])

t = near.merge(a1, on="file", how="left", validate="1:1")
assert t["cls"].notna().all() and not (set(t["file"]) & drop), "master/near-dup/review lists disagree"
assert set(t["label"]) == set(json.load(open(META / "classes.json")).values()), "class names differ from classes.json"

# photo sessions: dedup groups that belong to the same home photo session get merged
SESSION_MERGES = [("rice/rice_0128.jpg", "rice/rice_0157.jpg")]      # red tray + red bowl, same cloth and day
for a, b in SESSION_MERGES:
    ga = t.loc[t["file"] == a, "group_id"].iat[0]; gb = t.loc[t["file"] == b, "group_id"].iat[0]
    t.loc[t["group_id"] == gb, "group_id"] = ga
t["group_size"] = t.groupby("group_id")["file"].transform("size")
t["session"] = np.where(t["group_size"] >= 15, t["group_id"], "")      # big groups = photo sessions

box = set(pd.read_csv(META / "black_boxes.csv")["file"])
lb = pd.read_csv(META / "letterbox.csv").set_index("file")[["x0", "y0", "x1", "y1"]]
t["has_box"] = t["file"].isin(box)
t = t.merge(lb, left_on="file", right_index=True, how="left")
t["letterbox"] = t["x0"].notna()
t["note"] = t["file"].map(rv[rv.action == "note"].set_index("file")["detail"]).fillna("")
num = t["file"].str.extract(r"_(\d+)\.jpg$")[0].astype(int)
t["style"] = np.where((t["label"] == "pittu") & (num >= 33), "processed", "")   # HDR-looking pittu (visual review + blur score)
t["folder"] = t["file"].str.split("/").str[0]

cols = ["file", "label", "folder", "source", "group_id", "group_size", "session", "w", "h", "has_box", "letterbox",
        "x0", "y0", "x1", "y1", "style", "note", "flags", "nn_sim", "nn_file", "bytes", "sha1",
        "bright", "contrast", "blur", "chroma"]
t = t[cols].sort_values("file").reset_index(drop=True)
t.to_csv(META / "master.csv", index=False)

print(len(t), "images ->", META / "master.csv\n")
print(t.groupby("label").agg(images=("file", "size"), groups=("group_id", "nunique"), biggest=("group_size", "max"),
                             sessions=("session", lambda s: s[s != ""].nunique()),
                             boxes=("has_box", "sum"), crops=("letterbox", "sum")).to_string())
print("\nrelabelled:", t.loc[t["label"] != t["folder"], ["file", "label"]].values.tolist())
print("pittu with processed style:", int((t["style"] == "processed").sum()), "of", int((t["label"] == "pittu").sum()))
print("sessions:", t[t["session"] != ""].groupby("session").agg(n=("file", "size"), first=("file", "min"), last=("file", "max")).to_string())

1357 images -> data\meta\master.csv

                images  groups  biggest  sessions  boxes  crops
label                                                          
dosa               176     164        3         0      0      2
hoppers            156     118        3         0      2     32
idli                85      84        2         0      0      2
kottu               66      59        3         0      1      0
milk_rice          143     108        3         0      2     31
paratha            114     101        3         0      0      4
pittu              246     209        5         0      0      3
rice               288     161       49         3     26      0
string_hoppers      83      78        3         0      0      1

relabelled: [['dosa/dosa_0105.jpg', 'idli']]
pittu with processed style: 214 of 246
sessions:           n               first                last
session                                            
g0852    16  rice/rice_0011.jpg  rice/rice_0026.jpg
g0853   

In [4]:
# B-2 build data/imgs_prepared  (drops applied, letterbox cropped, black boxes inpainted, long side <= 1024)
import os, ast, shutil
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
import cv2
from scipy import ndimage

root = Path.cwd()
while not (root / "data" / "imgs_cleaned").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
CLEAN = Path("data/imgs_cleaned"); META = Path("data/meta")
PREP = Path("data/imgs_prepared"); OUT = Path("logs/preprocess_b"); (OUT / "sheets").mkdir(parents=True, exist_ok=True)
MAXSIDE = 1024

t = pd.read_csv(META / "master.csv").drop(columns=["path", "pw", "ph", "inpainted", "left", "suspect"], errors="ignore")
boxes = {r.file: ast.literal_eval(r.boxes) for r in pd.read_csv(META / "black_boxes.csv").itertuples()}

# boxes the detectors cannot see because they touch a dark border: (x0, y0, x1, y1) as fractions of the image
MANUAL = {"rice/rice_0280.jpg": [(0.00, 0.64, 0.14, 0.78)]}

def box_mask(W, H, bxs, pad=4):
    s = min(1.0, 320 / W, 320 / H)                   # the detector worked on a <=320 px thumbnail
    mask = np.zeros((H, W), np.uint8)
    for x, y, w, h in bxs:
        x0, y0 = max(int(x / s) - pad, 0), max(int(y / s) - pad, 0)
        x1, y1 = min(int(np.ceil((x + w) / s)) + pad, W), min(int(np.ceil((y + h) / s)) + pad, H)
        mask[y0:y1, x0:x1] = 255
    return mask

def black_mask(a, thr=10, min_area=20, max_frac=0.03, fill=0.6, grow=3):
    """pure-black blobs of box size, also overlapping boxes and slivers cut by the image edge"""
    H, W = a.shape[:2]
    lab, n = ndimage.label(a.max(axis=2) <= thr)
    m = np.zeros((H, W), np.uint8)
    for i, sl in enumerate(ndimage.find_objects(lab), 1):
        if sl is None: continue
        comp = lab[sl] == i
        area = int(comp.sum()); h, w = comp.shape
        if area < min_area or area > max_frac * H * W or area / (h * w) < fill: continue
        m[sl][comp] = 255
    return cv2.dilate(m, np.ones((2 * grow + 1, 2 * grow + 1), np.uint8))

def prepare(r):
    im = Image.open(CLEAN / r.file).convert("RGB")
    inp = r.file in boxes or r.file in MANUAL
    if inp:                                           # inpaint on the full image, before any crop
        a = np.asarray(im).copy()
        bm = box_mask(im.width, im.height, boxes[r.file]) if r.file in boxes else np.zeros((im.height, im.width), np.uint8)
        for x0, y0, x1, y1 in MANUAL.get(r.file, []):
            bm[int(y0 * im.height):int(y1 * im.height), int(x0 * im.width):int(x1 * im.width)] = 255
        im = Image.fromarray(cv2.inpaint(a, np.maximum(bm, black_mask(a)), 5, cv2.INPAINT_TELEA))
    if r.letterbox:
        im = im.crop((int(r.x0), int(r.y0), int(r.x1), int(r.y1)))
    if max(im.size) > MAXSIDE:
        s = MAXSIDE / max(im.size)
        im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
    return im, inp

if PREP.exists(): shutil.rmtree(PREP)                 # derived folder, always rebuilt from scratch
rows = []
for r in tqdm(list(t.itertuples(index=False))):
    im, inp = prepare(r)
    dst = PREP / r.label / Path(r.file).name
    dst.parent.mkdir(parents=True, exist_ok=True)
    im.save(dst, quality=95, subsampling=0)
    rows.append(dict(file=r.file, path=f"{r.label}/{Path(r.file).name}", pw=im.width, ph=im.height, inpainted=inp))
t = t.merge(pd.DataFrame(rows), on="file", validate="1:1")
t.to_csv(META / "master.csv", index=False)

n_out = sum(1 for _ in PREP.rglob("*.jpg"))
assert n_out == len(t), f"{n_out} files written, expected {len(t)}"
t["ar"] = t[["pw", "ph"]].max(axis=1) / t[["pw", "ph"]].min(axis=1)
print(f"{n_out} images in {PREP} | cropped: {int(t.letterbox.sum())} | inpainted: {int(t.inpainted.sum())} | "
      f"downscaled (>{MAXSIDE}px): {int((t[['w', 'h']].max(axis=1) > MAXSIDE).sum())}")
print(t.groupby("label").agg(images=("file", "size"), min_side=("pw", "min"), ar_median=("ar", "median"),
                             ar_gt_1_2=("ar", lambda s: round((s > 1.2).mean(), 2)),
                             ar_gt_1_5=("ar", lambda s: round((s > 1.5).mean(), 2))).to_string())

def pair_sheet(items, path, cols=4, th=160):         # before | after
    nr = -(-len(items) // cols)
    sheet = Image.new("RGB", (cols * 2 * th, nr * (th + 14)), "white"); dr = ImageDraw.Draw(sheet)
    for k, (f, p) in enumerate(items):
        x, y = (k % cols) * 2 * th, (k // cols) * (th + 14)
        for j, src in enumerate((CLEAN / f, PREP / p)):
            im = Image.open(src).convert("RGB"); im.thumbnail((th - 4, th - 4)); sheet.paste(im, (x + j * th, y))
        dr.text((x + 2, y + th), Path(f).stem, fill="black")
    sheet.save(path, quality=90)

for p in (OUT / "sheets").glob("inpaint_*.jpg"): p.unlink()
ip = t[t.inpainted]
for i in range(0, len(ip), 24):
    pair_sheet(list(zip(ip.file.iloc[i:i + 24], ip.path.iloc[i:i + 24])), OUT / "sheets" / f"inpaint_{i // 24 + 1:02d}.jpg")
print("before/after sheets in", OUT / "sheets")

  0%|          | 0/1357 [00:00<?, ?it/s]

1357 images in data\imgs_prepared | cropped: 75 | inpainted: 31 | downscaled (>1024px): 7
                images  min_side  ar_median  ar_gt_1_2  ar_gt_1_5
label                                                            
dosa               176       630        1.0       0.01       0.00
hoppers            156       359        1.0       0.28       0.08
idli                85       408        1.0       0.02       0.01
kottu               66       349        1.0       0.48       0.20
milk_rice          143       419        1.0       0.29       0.07
paratha            114       361        1.0       0.03       0.00
pittu              246       630        1.0       0.01       0.01
rice               288       640        1.0       0.00       0.00
string_hoppers      83       480        1.0       0.01       0.00
before/after sheets in logs\preprocess_b\sheets
